# Feature Engineering - Customers Dataset

## Objective

The objective of this notebook is to transform the raw customer dataset into a machine learning-ready dataset by creating meaningful features, improving data representation, and preparing the data for predictive modeling.

The engineered features are designed to capture customer demographics, account history, registration behavior, and loyalty characteristics that can improve the performance of downstream machine learning models such as Customer Churn Prediction, Customer Lifetime Value (CLV) Prediction, and Customer Segmentation.

2. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from datetime import datetime

pd.set_option("display.max_columns", None)

3. Load Dataset

In [2]:
df = pd.read_csv('/content/datasets/customers.csv')

4. Dataset Overview

In [3]:
df.head()

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,loyalty_points
0,1,Arjun,Modi,arjun.modi1@gmail.com,9923974559,Female,1971-08-08,2021-03-10,3729
1,2,Riya,Kapoor,riya.kapoor2@gmail.com,9922842611,Female,1978-01-04,2021-09-04,2326
2,3,Neha,Mehta,neha.mehta3@gmail.com,9758661182,Female,1981-07-04,2021-01-09,1842
3,4,Aarav,Trivedi,aarav.trivedi4@gmail.com,9228681267,Female,1995-08-17,2024-11-09,1116
4,5,Neha,Desai,neha.desai5@gmail.com,9251875017,Male,1992-10-17,2022-05-29,3115


In [4]:
df.shape

(5000, 9)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   customer_id        5000 non-null   int64 
 1   first_name         5000 non-null   object
 2   last_name          5000 non-null   object
 3   email              5000 non-null   object
 4   phone              5000 non-null   int64 
 5   gender             5000 non-null   object
 6   birth_date         5000 non-null   object
 7   registration_date  5000 non-null   object
 8   loyalty_points     5000 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 351.7+ KB


5. Convert Date Columns

In [6]:
df["birth_date"] = pd.to_datetime(df["birth_date"])

df["registration_date"] = pd.to_datetime(df["registration_date"])

6. Create Reference Date

In [7]:
REFERENCE_DATE = pd.Timestamp("2026-01-01")

7. Feature Engineering

In [10]:
# Feature 1 - Customer Age
df["age"] = (
    REFERENCE_DATE - df["birth_date"]
).dt.days // 365

In [11]:
# Feature 2 - Age Group
bins = [18,25,35,45,60,100]

labels = [
    "18-25",
    "26-35",
    "36-45",
    "46-60",
    "60+"
]

df["age_group"] = pd.cut(
    df["age"],
    bins=bins,
    labels=labels
)

In [12]:
# Feature 3 - Customer Tenure (Days)
df["customer_tenure_days"] = (
    REFERENCE_DATE -
    df["registration_date"]
).dt.days

In [13]:
# Feature 4 - Customer Tenure (Months)
df["customer_tenure_months"] = (
    df["customer_tenure_days"] / 30
).round(1)

In [14]:
# Feature 5 - Customer Tenure (Years)
df["customer_tenure_years"] = (
    df["customer_tenure_days"] / 365
).round(2)

In [15]:
# Feature 6 - Registration Year
df["registration_year"] = (
    df["registration_date"].dt.year
)

In [16]:
# Feature 7 - Registration Quarter
df["registration_quarter"] = (
    df["registration_date"].dt.quarter
)

In [17]:
# Feature 8 - Registration Month
df["registration_month"] = (
    df["registration_date"].dt.month_name()
)

In [18]:
# Feature 9 - Registration Weekday
df["registration_weekday"] = (
    df["registration_date"].dt.day_name()
)

In [20]:
# Feature 10 - Registration Weekend Flag
df["registered_on_weekend"] = (
    df["registration_date"].dt.weekday >= 5
).astype(int)

In [21]:
# Feature 11 - Loyalty Segment
bins = [0,100,500,1000,5000]

labels = [
    "Low",
    "Medium",
    "High",
    "VIP"
]

df["loyalty_segment"] = pd.cut(
    df["loyalty_points"],
    bins=bins,
    labels=labels
)

In [22]:
# Feature 12 - Loyalty Tier
tier = {
    "Low":1,
    "Medium":2,
    "High":3,
    "VIP":4
}

df["loyalty_tier"] = (
    df["loyalty_segment"]
      .map(tier)
)

In [23]:
# Feature 13 - Senior Citizen Flag
df["is_senior"] = (
    df["age"] >= 60
).astype(int)

In [24]:
# Feature 14 - Young Customer Flag
df["is_young_customer"] = (
    df["age"] <= 25
).astype(int)

In [25]:
# Feature 15 - Adult Customer Flag
df["is_adult"] = (
    (df["age"] >= 26) &
    (df["age"] <= 45)
).astype(int)

In [26]:
# Feature 16 - Long-Term Customer Flag
df["is_long_term_customer"] = (
    df["customer_tenure_years"] >= 5
).astype(int)

In [27]:
# Feature 17 - New Customer Flag
df["is_new_customer"] = (
    df["customer_tenure_days"] <= 365
).astype(int)

In [28]:
# Feature 18 - Loyalty Points per Year
df["loyalty_points_per_year"] = (
    df["loyalty_points"] /
    (df["customer_tenure_years"] + 0.01)
).round(2)

In [29]:
# Feature 19 - Email Domain
df["email_domain"] = (
    df["email"]
      .str.split("@")
      .str[-1]
      .str.lower()
)

8. Review Newly Created Features

In [30]:
new_features = [
    "age",
    "age_group",
    "customer_tenure_days",
    "customer_tenure_months",
    "customer_tenure_years",
    "registration_year",
    "registration_quarter",
    "registration_month",
    "registration_weekday",
    "registered_on_weekend",
    "loyalty_segment",
    "loyalty_tier",
    "is_senior",
    "is_young_customer",
    "is_adult",
    "is_long_term_customer",
    "is_new_customer",
    "loyalty_points_per_year",
    "email_domain"
]

df[new_features].head()

,age,age_group,customer_tenure_days,customer_tenure_months,customer_tenure_years,registration_year,registration_quarter,registration_month,registration_weekday,registered_on_weekend,loyalty_segment,loyalty_tier,is_senior,is_young_customer,is_adult,is_long_term_customer,is_new_customer,loyalty_points_per_year,email_domain
0,54,46-60,1758,58.6,4.82,2021,1,March,Wednesday,0,VIP,4,0,0,0,0,0,772.05,gmail.com
1,48,46-60,1580,52.7,4.33,2021,3,September,Saturday,1,VIP,4,0,0,0,0,0,535.94,gmail.com
2,44,36-45,1818,60.6,4.98,2021,1,January,Saturday,1,VIP,4,0,0,1,0,0,369.14,gmail.com
3,30,26-35,418,13.9,1.15,2024,4,November,Saturday,1,VIP,4,0,0,1,0,0,962.07,gmail.com
4,33,26-35,1313,43.8,3.60,2022,2,May,Sunday,1,VIP,4,0,0,1,0,0,862.88,gmail.com


9. Validate Engineered Features

In [31]:
df[new_features].describe(include="all")

,age,age_group,customer_tenure_days,customer_tenure_months,customer_tenure_years,registration_year,registration_quarter,registration_month,registration_weekday,registered_on_weekend,loyalty_segment,loyalty_tier,is_senior,is_young_customer,is_adult,is_long_term_customer,is_new_customer,loyalty_points_per_year,email_domain
count,5000.000000,5000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000,5000,5000.000000,5000,5000.0,5000.0,5000.000000,5000.000000,5000.000000,5000.00000,5000.0000,5000
unique,NaN,4,NaN,NaN,NaN,NaN,NaN,12,7,NaN,4,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1
top,NaN,36-45,NaN,NaN,NaN,NaN,NaN,March,Saturday,NaN,VIP,4.0,NaN,NaN,NaN,NaN,NaN,NaN,gmail.com
freq,NaN,1442,NaN,NaN,NaN,NaN,NaN,491,736,NaN,3998,3998.0,NaN,NaN,NaN,NaN,NaN,NaN,5000
mean,37.666000,NaN,1057.513200,35.250260,2.897238,2022.626200,2.436800,NaN,NaN,0.287400,NaN,NaN,0.0,0.157000,0.565000,0.156000,0.19680,inf,NaN
std,10.258689,NaN,659.758513,21.991419,1.807542,1.805416,1.129717,NaN,NaN,0.452595,NaN,NaN,0.0,0.363837,0.495807,0.362892,0.39762,NaN,NaN
min,20.000000,NaN,-108.000000,-3.600000,-0.300000,2020.000000,1.000000,NaN,NaN,0.000000,NaN,NaN,0.0,0.000000,0.000000,0.000000,0.00000,-440600.0000,NaN
25%,29.000000,NaN,484.750000,16.175000,1.330000,2021.000000,1.000000,NaN,NaN,0.000000,NaN,NaN,0.0,0.000000,0.000000,0.000000,0.00000,363.5375,NaN
50%,38.000000,NaN,1063.000000,35.400000,2.910000,2023.000000,2.000000,NaN,NaN,0.000000,NaN,NaN,0.0,0.000000,1.000000,0.000000,0.00000,778.6800,NaN
75%,46.000000,NaN,1629.000000,54.300000,4.460000,2024.000000,3.000000,NaN,NaN,1.000000,NaN,NaN,0.0,0.000000,1.000000,0.000000,0.00000,1523.8700,NaN


In [32]:
df[new_features].isnull().sum()

,0
age,0
age_group,0
customer_tenure_days,0
customer_tenure_months,0
customer_tenure_years,0
registration_year,0
registration_quarter,0
registration_month,0
registration_weekday,0
registered_on_weekend,0


10. Save Feature Engineered Dataset

In [37]:
df.to_csv(
    "datasets/customers_feature_engineered.csv",
    index=False
)

In [38]:
df_processed = pd.read_csv('/content/datasets/customers_feature_engineered.csv')

In [39]:
df_processed

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,loyalty_points,age,age_group,customer_tenure_days,customer_tenure_months,customer_tenure_years,registration_year,registration_quarter,registration_month,registration_weekday,registered_on_weekend,loyalty_segment,loyalty_tier,is_senior,is_young_customer,is_adult,is_long_term_customer,is_new_customer,loyalty_points_per_year,email_domain
0,1,Arjun,Modi,arjun.modi1@gmail.com,9923974559,Female,1971-08-08,2021-03-10,3729,54,46-60,1758,58.6,4.82,2021,1,March,Wednesday,0,VIP,4,0,0,0,0,0,772.05,gmail.com
1,2,Riya,Kapoor,riya.kapoor2@gmail.com,9922842611,Female,1978-01-04,2021-09-04,2326,48,46-60,1580,52.7,4.33,2021,3,September,Saturday,1,VIP,4,0,0,0,0,0,535.94,gmail.com
2,3,Neha,Mehta,neha.mehta3@gmail.com,9758661182,Female,1981-07-04,2021-01-09,1842,44,36-45,1818,60.6,4.98,2021,1,January,Saturday,1,VIP,4,0,0,1,0,0,369.14,gmail.com
3,4,Aarav,Trivedi,aarav.trivedi4@gmail.com,9228681267,Female,1995-08-17,2024-11-09,1116,30,26-35,418,13.9,1.15,2024,4,November,Saturday,1,VIP,4,0,0,1,0,0,962.07,gmail.com
4,5,Neha,Desai,neha.desai5@gmail.com,9251875017,Male,1992-10-17,2022-05-29,3115,33,26-35,1313,43.8,3.60,2022,2,May,Sunday,1,VIP,4,0,0,1,0,0,862.88,gmail.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,Ananya,Patel,ananya.patel4996@gmail.com,9128721902,Male,1979-07-09,2020-10-25,3643,46,46-60,1894,63.1,5.19,2020,4,October,Sunday,1,VIP,4,0,0,0,1,0,700.58,gmail.com
4996,4997,Diya,Joshi,diya.joshi4997@gmail.com,9923475768,Male,1976-10-18,2021-09-06,582,49,46-60,1578,52.6,4.32,2021,3,September,Monday,0,High,3,0,0,0,0,0,134.41,gmail.com
4997,4998,Ananya,Joshi,ananya.joshi4998@gmail.com,9319516293,Male,1990-02-07,2021-03-09,3848,35,26-35,1759,58.6,4.82,2021,1,March,Tuesday,0,VIP,4,0,0,1,0,0,796.69,gmail.com
4998,4999,Aditya,Patel,aditya.patel4999@gmail.com,9929205533,Female,1995-03-26,2024-04-30,2324,30,26-35,611,20.4,1.67,2024,2,April,Tuesday,0,VIP,4,0,0,1,0,0,1383.33,gmail.com
